# Ensemble Learning : Bagging & Its Variants
 just practising different bagging techniques and understanding the concept of wisdom of the crowd.        
implemented all these techniques on a toy dataset generated using make_classification 
- **Bagging** : Row sampling **with replacement** (same row can repeat)
- **Pasting** : Row sampling **without replacement** (all rows unique per sample)
- **Random Subspaces** : **Column based sampling** each model sees a different subset of features, all rows used
- **Random Patches** : **Both row and column sampling**  most aggressive randomization, best for high dimensional data        

- **OOB Score** — Free validation using the ~37% data each tree never sees



In [55]:
from sklearn.datasets import make_classification 
from sklearn.metrics import accuracy_score
import pandas as pd 
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split


In [56]:
X,y = make_classification(n_samples=10000, n_features=10,n_informative=3)

In [57]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [58]:
dt= DecisionTreeClassifier(random_state=42)
dt.fit(X_train,y_train)


DecisionTreeClassifier(random_state=42)

In [59]:
y_pred = dt.predict(X_test)
y_pred.shape

(2000,)

In [60]:
accuracy_score(y_test,y_pred)

0.94

# Bagging (Row sampling)
## using Decision Tree Classifier Estimator
currently we are focusing on row sampling as of now

In [61]:
bag= BaggingClassifier(estimator=DecisionTreeClassifier(),n_estimators=500,max_samples=0.25,bootstrap=True,random_state=42)

here bootstrap = true simply means that we can have duplicate entries as training points for our base models
- it is the core difference bw normal row sampling bagging technique and pasting

In [62]:
bag.fit(X_train,y_train)

BaggingClassifier(estimator=DecisionTreeClassifier(), max_samples=0.25,
                  n_estimators=500, random_state=42)

In [63]:
y_pred1=bag.predict(X_test)
accuracy_score(y_test,y_pred1)

0.956

## Using SVC Estimator

In [64]:
bagsvc= BaggingClassifier(estimator=SVC(),n_estimators=500,max_samples=0.25,bootstrap=True,random_state=42)

In [65]:
bagsvc.fit(X_train,y_train)

BaggingClassifier(estimator=SVC(), max_samples=0.25, n_estimators=500,
                  random_state=42)

In [66]:
y_svc=bagsvc.predict(X_test)

In [67]:
accuracy_score(y_test,y_svc)

0.923

# Pasting 
- no duplicate rows allowed in test set
- simply just set bootstrap = false

In [69]:
bag = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=500,
    max_samples=0.25,
    bootstrap=False,
    random_state=42,
    verbose = 1,
    n_jobs=-1
)

In [70]:
bag.fit(X_train,y_train)


[Parallel(n_jobs=10)]: Using backend LokyBackend with 10 concurrent workers.
[Parallel(n_jobs=10)]: Done   2 out of  10 | elapsed:    4.9s remaining:   19.5s
[Parallel(n_jobs=10)]: Done  10 out of  10 | elapsed:    5.0s finished


BaggingClassifier(bootstrap=False, estimator=DecisionTreeClassifier(),
                  max_samples=0.25, n_estimators=500, n_jobs=-1,
                  random_state=42, verbose=1)

In [ ]:
y_pred = bag.predict(X_test)

[Parallel(n_jobs=10)]: Using backend LokyBackend with 10 concurrent workers.
[Parallel(n_jobs=10)]: Done   2 out of  10 | elapsed:    0.1s remaining:    0.3s
[Parallel(n_jobs=10)]: Done  10 out of  10 | elapsed:    0.2s finished


In [72]:
print("Pasting classifier",accuracy_score(y_test,y_pred))

Pasting classifier 0.9575


## Random Subspaces 
- columns based sampling
- base models trained on different set of features each time
- simply just specify max_features=0.5 (note that here we are not doing row sampling)

In [75]:
bag = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=500,
    max_samples=1.0,
    bootstrap=False,
    max_features=0.5,
    bootstrap_features=True,
    random_state=42
)

In [76]:
bag.fit(X_train,y_train)

BaggingClassifier(bootstrap=False, bootstrap_features=True,
                  estimator=DecisionTreeClassifier(), max_features=0.5,
                  n_estimators=500, random_state=42)

In [77]:
y_pred = bag.predict(X_test)

In [78]:
accuracy_score(y_test,y_pred)

0.9515

## Random Patches
- both row and column sampling
- max_features=0.5, max_sample=0.25 

In [83]:
bag = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=500,
    max_samples=0.25,
    bootstrap=True,
    max_features=0.5,
    bootstrap_features=True,
    random_state=42
)

In [85]:
bag.fit(X_train,y_train)


BaggingClassifier(bootstrap_features=True, estimator=DecisionTreeClassifier(),
                  max_features=0.5, max_samples=0.25, n_estimators=500,
                  random_state=42)

In [87]:
y_pred = bag.predict(X_test)

In [88]:
print("Random Patches classifier",accuracy_score(y_test,y_pred))

Random Patches classifier 0.9405


## OOB Score
out of bag score?
- lets try to understand what's actually oob
- Every tree in Bagging is trained on a **bootstrap sample**
- Bootstrap = random rows picked **with replacement**
- Meaning the same row can appear multiple times in one sample
### The 63% Rule
it's a direct consequence of one of the most fundamental constants in mathematics (e)
- When we train a Bagging model, each tree is trained on a bootstrap sample (random rows picked with replacement). This means some rows naturally never get picked for a particular tree 
- It is **statistically proven** (via Euler's number `e`) that:
    - 63% of data is seen by each tree (used for training)
    - 37% of data is never seen by that tree (OOB samples)
- even after this entire randomness ? yes it still stays unseen and thaths 63/37 rule that only 63 percent of data is seen.     
       
P(never picked) = $(1 - 1/N)^N → 1/e ≈ 0.368$ as N grows

### so what? what are we supposed to do with this rule?
- we can use the unseen point to evalute our model and understand how it performs on unseen data points 
- Take majority vote and compare with actual label
- This gives us **OOB Score** a free, unbiased measure of model performance


In [92]:
bag = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=500,
    max_samples=0.25,
    bootstrap=True,
    oob_score=True,
    random_state=42,
    verbose = 1,
    n_jobs=-1
)

In [93]:
bag.fit(X_train,y_train)

[Parallel(n_jobs=10)]: Using backend LokyBackend with 10 concurrent workers.
[Parallel(n_jobs=10)]: Done   2 out of  10 | elapsed:    4.4s remaining:   17.6s
[Parallel(n_jobs=10)]: Done  10 out of  10 | elapsed:    4.6s finished


BaggingClassifier(estimator=DecisionTreeClassifier(), max_samples=0.25,
                  n_estimators=500, n_jobs=-1, oob_score=True, random_state=42,
                  verbose=1)

In [94]:
bag.oob_score_

0.9585

In [95]:
y_pred = bag.predict(X_test)
accuracy_score(y_test,y_pred)

[Parallel(n_jobs=10)]: Using backend LokyBackend with 10 concurrent workers.
[Parallel(n_jobs=10)]: Done   2 out of  10 | elapsed:    0.1s remaining:    0.4s
[Parallel(n_jobs=10)]: Done  10 out of  10 | elapsed:    0.2s finished


0.956

# Bagging Tips
- Bagging generally gives better results than Pasting
- Good results come around the 25% to 50% row sampling mark
- Random patches and subspaces should be used while dealing with high dimensional data
- To find the correct hyperparameter values we can do GridSearchCV/RandomSearchCV

> **Key Insight** — A single Decision Tree gave 87.5% accuracy.
> Simply wrapping it in a BaggingClassifier pushed it to 91.6%